In [1]:
%reload_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os
from typing import Dict, List,Tuple,Any
import yaml
import ast
import re
import itertools

In [2]:
%cd ..

/home/ssivanes/Fgpt


/data/ssivanes/fparser-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Processor class
from processor import Processor
from extractor import Extractor
from isolator import Isolator
processor = Processor()

INFO     Processor initialized.

In [4]:
rest_of_path = "/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol"
work = os.getenv("work")

In [5]:
isolator = Isolator(rest_of_path, target_module, work,False)

cls = Extractor(isolator.module_dir_sp, isolator.module_tree_sp)
cls.find_subroutines()
cls.extract_loop_indices()

INFO     Processor initialized.

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90

INFO     Successfully parsed string!

INFO     Processor initialized.

In [6]:
cls.subroutine_keys_all

{'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag',
 'hydrol_split_soil',
 'hydrol_tmc_update',
 'hydrol_vegupd'}

In [7]:
cls.subroutine_keys_ncl

{'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag',
 'hydrol_split_soil',
 'hydrol_tmc_update'}

In [8]:
# FInding the parents but also using the 
print(cls.subroutine_keys_all - cls.subroutine_keys_ncl)

{'hydrol_soil', 'hydrol_vegupd', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_muff_radial_coef_setup'}


In [9]:
subroutine_key = 'hydrol_vegupd'

In [10]:
subroutine_tree = cls.subroutines[subroutine_key]
cls.find_variables(subroutine_tree, subroutine_key)
cls.extract_names(subroutine_key)

In [11]:
cls.call_within_sub

defaultdict(set,
            {'hydrol_main': {'explicitsnow_main',
              'histwrite_p',
              'hydrol_alma',
              'hydrol_canop',
              'hydrol_flood',
              'hydrol_hydraulic_arch_tuzet_calc',
              'hydrol_nudge_mc_diag',
              'hydrol_nudge_snow',
              'hydrol_soil',
              'hydrol_vegupd'},
             'hydrol_vegupd': {'hydrol_tmc_update'},
             'hydrol_soil': {'hydrol_diag_soil',
              'hydrol_diag_soil_flux',
              'hydrol_nudge_mc',
              'hydrol_root_profile',
              'hydrol_soil_coef',
              'hydrol_soil_froz',
              'hydrol_soil_infilt',
              'hydrol_soil_setup',
              'hydrol_soil_smooth_over_mcs2',
              'hydrol_soil_smooth_under_mcr',
              'hydrol_soil_tridiag',
              'hydrol_split_soil'},
             'hydrol_nudge_snow': {'flinget',
              'flininfo',
              'scatter',
              'xios

In [12]:
for child_subroutine_key in list(cls.call_within_sub[subroutine_key]):
    child_subroutine_tree = cls.subroutines[child_subroutine_key]
    isolator.child_subroutine_call[subroutine_key].append(child_subroutine_tree)

In [13]:
print(len(isolator.child_subroutine_call[subroutine_key]))

1


In [14]:
for child_subroutine_key in list(cls.call_within_sub[subroutine_key]):
    if child_subroutine_key not in cls.call_within_sub: # THIS line is just to check if the child itself is not a parent of another 
        mod_child_subroutine_tree, error_flag = isolator.isolate_child_subroutine(cls, child_subroutine_key, cls.var_local_names[subroutine_key])
        if mod_child_subroutine_tree is not None:
            isolator.child_subroutine_call[subroutine_key].append(mod_child_subroutine_tree)
        if error_flag is not None:
            isolator.child_error_flag[subroutine_key][child_subroutine_key] = error_flag
        isolator.collect_global_vars_decl(cls.dec_global[child_subroutine_key], cls.dec_global[subroutine_key])


INFO     Trying to isolate a child subroutine .... hydrol_tmc_update

INFO     Processor initialized.

INFO     Successfully parsed string!

INFO     Processor initialized.

INFO     ⏳... Searching for variable 'humtot'

...

INFO     Compilation process completed!

INFO     Benchmark files exist. Now running the unit tests ...

 --- inside the main program ---
 --- add the declaration and initialization in module global ---
 --- inside the read dummy routine for hydrol_tmc_update ---
 Execution time :    1.2819999999999999E-003


INFO     Execution completed in /home/ssivanes/Fgpt/hydrol/hydrol_tmc_update

INFO     Successfully wrote code to file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

In [15]:
print(list(cls.dec_global[subroutine_key])) # THis will now contain all the global values that will be used inside the 
# Parent subroutine and global values of the child subroutines. that we retrieve using this isolator.collect_global_vars_decl(cls.dec_global[child_subroutine_key], cls.dec_global[subroutine_key])
# which modifies direclty the cls.dec_global[subroutine_key] of the parent declarations 

['humtot', 'vegtot_old', 'resdist', 'pref_soil_veg', 'dz', 'numout', 'min_sechiba', 'printlev', 'water2infilt', 'ipslerr_p', 'trois', 'vegtot', 'tmc', 'mc', 'huit', 'zero']


In [16]:
cls.extract_intent(subroutine_key, subroutine_tree, cls.call_within_sub[subroutine_key])
cls.clean_subroutine(subroutine_key, subroutine_tree)
code_string = subroutine_tree.tofortran()
working_tree = Processor().parse_fortran_string(code_string)

WARNING  Incorrect intent for frac_bare. Expected: INOUT, Found: OUT. Correct it!

WARNING  Original Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(OUT) :: frac_bare

INFO     Modified Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(INOUT) :: frac_bare

INFO     Processor initialized.

INFO     Successfully parsed string!

In [17]:
cls.find_variables(subroutine_tree, subroutine_key)
cls.extract_names(subroutine_key)
cls.var_global[subroutine_key] = cls.var_global[subroutine_key] - cls.call_within_sub[subroutine_key] - set(cls.dec_global[subroutine_key].keys())
cls.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, cls.var_global[subroutine_key], subroutine_key)

INFO     Processor initialized.

INFO     ⏳... Searching for variable 'un'

INFO     Module 'ioipsl' is added into the queue.

INFO     Module 'xios_orchidee' is added into the queue.

INFO     Module 'constantes' is added into the queue.

...

INFO     INTEGER(KIND = i_std), ALLOCATABLE, SAVE, DIMENSION(:, :) :: mask_soiltile

INFO     'mask_soiltile' is found in 'hydrol_init' of the module 'hydrol'

INFO     ALLOCATE(mask_soiltile(kjpindex, nstm), STAT = ier)

INFO     The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/

INFO     ✅ Variable found!

In [18]:
cls.var_global[subroutine_key]

{'frac_bare_ns',
 'mask_soiltile',
 'mask_veget',
 'ok_bare_soil_new',
 'un',
 'vegetmax_soil'}

In [19]:
print(list(cls.dec_global[subroutine_key])) # THis is the parent which contains the global variables of it's own and that of it's chidlren
print(list(cls.dec_global[child_subroutine_key]),child_subroutine_key)
print(set(cls.dec_global[subroutine_key])- set(cls.dec_global[child_subroutine_key]))

['humtot', 'vegtot_old', 'resdist', 'pref_soil_veg', 'dz', 'numout', 'min_sechiba', 'printlev', 'water2infilt', 'ipslerr_p', 'trois', 'vegtot', 'tmc', 'mc', 'huit', 'zero', 'un', 'vegetmax_soil', 'mask_veget', 'ok_bare_soil_new', 'frac_bare_ns', 'mask_soiltile']
['humtot', 'vegtot_old', 'resdist', 'pref_soil_veg', 'dz', 'numout', 'min_sechiba', 'printlev', 'water2infilt', 'ipslerr_p', 'trois', 'vegtot', 'tmc', 'mc', 'huit', 'zero'] hydrol_tmc_update
{'un', 'vegetmax_soil', 'mask_veget', 'ok_bare_soil_new', 'frac_bare_ns', 'mask_soiltile'}


In [20]:
cls.var_local_names[subroutine_key]

{'ji', 'jst', 'jv'}

In [21]:
cls.process_declaration_variables(cls.var_dummy[subroutine_key], subroutine_key)
for key in cls.dec_global[subroutine_key].keys():
    cls.process_declaration_variables(cls.dec_global[subroutine_key][key], subroutine_key)

shape_to_search = cls.shapes_variables[subroutine_key] - cls.scalar_variables[subroutine_key] - cls.var_global[subroutine_key]

if shape_to_search:
    cls.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, shape_to_search, subroutine_key)
    cls.var_global[subroutine_key].update(shape_to_search)

cls.extract_array_info(cls.dec_global[subroutine_key], cls.var_dummy[subroutine_key], subroutine_key)
cls.extract_loop_vect(subroutine_key, subroutine_tree)

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: humtot

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: resdist

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(nvm) :: pref_soil_veg

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dz

...

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mc

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: vegetmax_soil

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nvm) :: mask_veget

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nstm) :: mask_soiltile

In [22]:
cls.loop_dict

defaultdict(set,
            {'nslm': {'isl', 'jsl'},
             'kjpindex': {'ipts', 'ji'},
             'nvm': {'ivm', 'jv'},
             'nstm': {'ist', 'jst'},
             'itopmax': {'jsl'},
             'nslm - 1': {'jsl'},
             'imax - 1': {'ii'},
             'imin': {'ii'},
             '4': {'jsl'},
             'nslm - 2': {'jsl'},
             '2': {'jsl'},
             '1': {'jrp', 'jsl'},
             'nbp_glo': {'ji'},
             'nsnow': {'jg'},
             'nrp': {'jrp'},
             'nrp - 1': {'jrp'}})

In [23]:
cls.loop_vect

defaultdict(<function extractor.Extractor.__init__.<locals>.<lambda>()>,
            {'hydrol_tmc_update': 'DO ji = 1, kjpindex',
             'hydrol_vegupd': 'DO ji = 1, kjpindex'})

In [24]:
isolator.processor.add_declarations(
                cls.dec_global[subroutine_key],
                cls.var_modif_info[subroutine_key],
                openacc=isolator.openacc
                )

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dz

INFO     Successfully parsed statement:                                                                            
         IF (.NOT. ALLOCATED(dz)) THEN                                                                             
           ALLOCATE(dz(nslm), STAT = ier)                                                                          
         END IF

INFO     Successfully generated allocation statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns

INFO     Successfully parsed statement:                                                                            
         IF (.NOT. ALLOCATED(frac_bare_ns)) THEN                                                                   
           ALLOCATE(frac_bare_ns(kjpindex, nstm), STAT = ier)                                                      
         END IF

...

INFO     Successfully parsed statement:                                                                            
         READ(1363, IOSTAT = ier) vegtot_old                                                                       
         IF (ier /= 0) THEN                                                                                        
           WRITE(*, *) 'Error reading from file for vegtot_old. ', ' IOSTAT : ', ier                               
         END IF

INFO     Successfully parsed statement:                                                                            
         READ(1363, IOSTAT = ier) water2infilt                                                                     
         IF (ier /= 0) THEN                                                                                        
           WRITE(*, *) 'Error reading from file for water2infilt. ', ' IOSTAT : ', ier                             
         END IF

INFO     processing initialization completed!

INFO     Processing complete. PARAMETER elements sent to the left.

INFO     Declarations and allocations processed successfully

In [25]:
cls.var_dummy[subroutine_key]

[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')),))), Intent_Attr_Spec('INTENT', Intent_Spec('OUT')))), Entity_Decl_List(',', (Entity_Decl(Name('drain_upd'), None, None, None),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')), Explicit_Shape_Spec(None, Name('nvm'))))), Intent_Attr_Spec('INTENT', Intent_Spec('INOUT')))), Entity_Decl_List(',', (Entity_Decl(Name('frac_bare'), None, None, None),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')), Explicit_Shape_Spec(None, Name(

In [26]:
'ru_infilt' in list(cls.dec_global[subroutine_key].keys())

False

In [27]:
# Most of the elements are the retrieved variables are the same for the children as for the parent. But for the parent transformation
# we will primarily use the class format thus most of the previously created functions can be re used as well. This pratically means that 
# we need to first use global class template and fill it up then start adding the subroutines inside the class itself

In [28]:
%reload_ext autoreload
%autoreload 2
from transformer import Transformer
from utils import identify_replace_all

In [29]:
transformer = Transformer("/home/ssivanes/Fgpt/benchmark",isolator,cls,None,config_path = "/home/ssivanes/Fgpt/template.yaml")

In [30]:
# declaration_stmts = list(cls.dec_global[subroutine_key].values())
# ast_nodes = transformer.convert_SPECIFICATION_PART(declaration_stmts,cls_mode=True)


In [31]:
tree = transformer.update_global_python(subroutine_key,cls_mode = True,for_loop=True)
# THis the global module for the parent subroutine itself

INFO     Processor initialized.

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: humtot

INFO     Processor initialized.

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old

INFO     Processor initialized.

...

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nvm) :: mask_veget

INFO     Processor initialized.

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns

INFO     Processor initialized.

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nstm) :: mask_soiltile

In [32]:
print(ast.unparse(tree))

import numpy as np
import os
from scipy.io import FortranFile

class Global_module_hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.numout = np.int32(6)
        self.min_sechiba = np.float64(1e-08)
        self.printlev = np.int32(2)
        self.trois = np.float64(3.0)
        self.huit = np.float64(8.0)
        self.zero = np.float64(0.0)
        self.un = np.float64(1.0)
        self.ok_bare_soil_new = np.bool(False)
        self.dz = np.zeros((self.nslm,), dtype=np.float64)
        self.frac_bare_ns = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.humtot = np.zeros((self.kjpindex,), dt

In [33]:
print(transformer.dependant_variables)

{}


In [35]:
#cls.scalar_variables['hydrol_soil_smooth_under_mcr']

In [36]:
isolator.processor.reads_in_decleration_routine

[Execution_Part(Read_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Int_Literal_Constant('1363', None)), Io_Control_Spec('IOSTAT', Name('ier')))), None, Input_Item_List(',', (Name('ok_bare_soil_new'),))), If_Construct(If_Then_Stmt(Level_4_Expr(Name('ier'), '/=', Int_Literal_Constant('0', None))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Error reading from file for ok_bare_soil_new. '", None), Char_Literal_Constant("' IOSTAT : '", None), Name('ier')))), End_If_Stmt('IF', None)))]

In [37]:
print(walk(walk(isolator.processor.reads_in_read_routine,F23.Input_Item_List),F23.Name))

[Name('dz'), Name('frac_bare_ns'), Name('humtot'), Name('mask_soiltile'), Name('mask_veget'), Name('mc'), Name('pref_soil_veg'), Name('resdist'), Name('tmc'), Name('vegetmax_soil'), Name('vegtot'), Name('vegtot_old'), Name('water2infilt')]


In [38]:
# Now we need to create the main class which will contain all the child, parent subroutine, TO DO so we will use the empty global class template
# add remove the declaration_intialization method which is not needed 
import copy
main_class_template = transformer.out_module_python()

for node in ast.iter_child_nodes(main_class_template):
    if isinstance(node, ast.ClassDef):
        node.body = [
            item for item in node.body
            if not (isinstance(item, ast.FunctionDef) and item.name == "declaration_initialization")
        ]

"""
for node in ast.iter_child_nodes(main_class):
    if isinstance(node, ast.ClassDef):
        for fn in node.body:
            if isinstance(fn, ast.FunctionDef) and fn.name == "__init__":
                fn.body = [ast.Pass()]
"""
# And we modify it's name as well for this class 
# CHange the name as well to represent the parent class direclty
from utils import ast_walk

main_class_name = ast_walk(main_class_template,ast.ClassDef)
class_def = next(iter(main_class_name))
l = [subroutine_key[0].upper(), subroutine_key[1:]]
class_def.name = "".join(l)


# Now we need to ensure that the global values are also placed inside which is called composition
cls_info, import_nodes, instance_nodes = transformer.create_cls_info(tree)

# Use the instance_nodes to create the assign_node to be placed inside the __init__ as an arg and sent as argument
instance_node = instance_nodes[0]
name = instance_node.targets[0].id
instance_node.targets = [ast.Attribute(value = ast.Name(id='self',ctx=ast.Load()),
                                       attr = name,
                                       ctx = ast.Store())]

init_function_def = next(iter(ast_walk(class_def, ast.FunctionDef)))
transformer.add_instance(None,instance_nodes[0],cls_info,init_function_def,method_name = ["declaration_initialization"])

"""
for node in ast.iter_child_nodes(main_class_template):
    if isinstance(node, ast.ClassDef):
        for fn in node.body:
            if isinstance(fn, ast.FunctionDef) and fn.name == "__init__":
                # remove Pass nodes
                fn.body = [stmt for stmt in fn.body if not isinstance(stmt, ast.Pass)]
"""     
print(ast.unparse(ast.fix_missing_locations(main_class_template)))

import numpy as np
import os
from scipy.io import FortranFile

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()


In [39]:
def merge_instance_attributes(primary_dict, secondary_dict):
    """
    Merge attributes from primary and secondary dicts into instance-level view.
    
    - "self": all attributes from the primary class
    - "<instance>": only secondary attributes that are NOT already in primary
    """

    merged = {}

    for primary_module, classes in primary_dict.items():
        for primary_class, class_content in classes.items():
            # All primary attributes
            primary_attrs = class_content.get("attributes", {})
            merged["self"] = list(primary_attrs.keys())

            # Look inside secondary dict attributes
            for secondary_module, secondary_classes in secondary_dict.items():
                for secondary_class, composed_class in secondary_classes.items():
                    secondary_attrs = composed_class.get("attributes", {})

                    # Keep only attrs not in primary
                    unique_attrs = [
                        attr for attr in secondary_attrs.keys()
                        if attr not in primary_attrs
                    ]

                    merged[secondary_class] = unique_attrs

    return merged

tmp_class,_,_ = transformer.create_cls_info(class_def)
merged_class = merge_instance_attributes(tmp_class,cls_info)
print(merged_class)

{'self': ['nsnow', 'nslm', 'nvm', 'nstm', 'kjpindex', 'ier', 'ic0', 'ic', 'icr', 'start_time', 'stop_time'], 'gmhv': ['numout', 'min_sechiba', 'printlev', 'trois', 'huit', 'zero', 'un', 'ok_bare_soil_new', 'dz', 'frac_bare_ns', 'humtot', 'mask_soiltile', 'mask_veget', 'mc', 'pref_soil_veg', 'resdist', 'tmc', 'vegetmax_soil', 'vegtot', 'vegtot_old', 'water2infilt']}


In [40]:
# cls.var_dummy[subroutine_key]
declaration_stmts = [[elements] for elements in cls.var_dummy[subroutine_key]]
ast_nodes = transformer.convert_SPECIFICATION_PART(declaration_stmts,True,cls_mode=True)
assign_nodes = []
procedure_nodes = []
        
for node in ast_nodes:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        procedure_nodes.append(node)
    elif isinstance(node, (ast.Assign, ast.Assign)):
        assign_nodes.append(node)

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

...

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

In [41]:
print(procedure_nodes)

[]


In [42]:
for assign_node in assign_nodes:
    # print(assign_node.targets[0].attr)
    print(ast.unparse(assign_node))

self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
self.soiltile = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
self.veget = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
self.veget_max = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)


In [43]:
list(cls_info)[-1]

'Global_module_hydrol_vegupd'

In [44]:
transformer.global_state = False

In [45]:
transformer.insert_all_assign_nodes(assign_nodes,main_class_template,method_name='__init__',class_name=list(cls_info)[-1],method="declaration_initialization")

In [46]:
print(ast.unparse(ast.fix_missing_locations(main_class_template))) # After adding the attributes onto the __init__ method

import numpy as np
import os
from scipy.io import FortranFile

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soiltile = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.veget = np.zeros((self.kj

**Once we have created the template with the init method and the local variables of this parent class we just need to attack the subroutines itself then add to the body of the class itself and since these subroutines are instances of FUnctiondef every modification we make is directly taken into account inside the class itself**

In [47]:
# Now since we know that the child subrotuines are inside the isolator.child_subroutine_call['hydrol_vegupd'] which means we can just use the 
# the transformer.f2np.recursive_ast to transform this to python ast
module_stacks = []
for subroutines in isolator.child_subroutine_call[subroutine_key]:
    _,_,module_stack = transformer.f2np.recursive_ast(subroutines)
    if len(module_stack) > 1:
        raise ValueError(f'Length of module_stack larger than one')
    module_stacks.append(module_stack[-1])
    

In [48]:
print(ast.unparse(ast.fix_missing_locations(module_stacks[0]))) # childsubroutine

def hydrol_tmc_update(kjpindex, veget_max, soiltile, qsintveg, drain_upd, runoff_upd):
    vmr = np.zeros((kjpindex, nstm), dtype=np.float64)
    vmr_sum = np.zeros((kjpindex,), dtype=np.float64)
    delvegtot = np.zeros((kjpindex,), dtype=np.float64)
    mc_dilu = np.zeros((kjpindex, nslm), dtype=np.float64)
    infil_dilu = np.zeros((kjpindex,), dtype=np.float64)
    tmc_old = np.zeros((kjpindex, nstm), dtype=np.float64)
    water2infilt_old = np.zeros((kjpindex, nstm), dtype=np.float64)
    qsintveg_old = np.zeros((kjpindex, nvm), dtype=np.float64)
    test = np.zeros((kjpindex,), dtype=np.float64)
    mcaux = np.zeros((kjpindex, nslm, nstm), dtype=np.float64)
    for ji in range(0, kjpindex, 1):
        if vegtot_old[ji] > min_sechiba:
            for jv in range(0, nvm, 1):
                if veget_max[ji, jv] < min_sechiba and qsintveg[ji, jv] > 0.0:
                    jst = pref_soil_veg[jv]
                    if resdist[ji, jst] > zero:
                        index = jst
   

In [49]:
# NOw we do the transformation for the parent subroutine 
_,_,module_stack1 = transformer.f2np.recursive_ast(subroutine_tree) # parent subroutine 

In [50]:
parent_function_def = module_stack1[0]
print(ast.unparse(ast.fix_missing_locations(parent_function_def)))

def hydrol_vegupd(kjpindex, veget, veget_max, soiltile, qsintveg, frac_bare, drain_upd, runoff_upd):
    hydrol_tmc_update(kjpindex, veget_max, soiltile, qsintveg, drain_upd, runoff_upd)
    mask_veget[:, :] = 0
    mask_soiltile[:, :] = 0
    for jst in range(0, nstm, 1):
        for ji in range(0, kjpindex, 1):
            if soiltile[ji, jst] > min_sechiba:
                mask_soiltile[ji, jst] = 1
    for jv in range(0, nvm, 1):
        for ji in range(0, kjpindex, 1):
            if veget_max[ji, jv] > min_sechiba:
                mask_veget[ji, jv] = 1
    vegetmax_soil[:, :, :] = zero
    for jv in range(0, nvm, 1):
        jst = pref_soil_veg[jv]
        for ji in range(0, kjpindex, 1):
            if mask_soiltile[ji, jst] > 0 and vegtot[ji] > min_sechiba:
                vegetmax_soil[ji, jv, jst] = veget_max[ji, jv] / soiltile[ji, jst]
    for ji in range(0, kjpindex, 1):
        if veget_max[ji, 1] > min_sechiba:
            frac_bare[ji, 1] = un
        else:
            

In [51]:
module_name = list(cls_info.keys())[-1]
instance_name = list(cls_info[module_name].keys())[-1]

In [52]:
# since the args are entirely present within both of the subroutines are present are inside the class itself so we just simply replace
# the args by self and the instance of the global values itself
child_function_def = module_stack[0]

In [53]:
# Now we add them inside the class as methods 
class_def.body.extend(module_stacks + [parent_function_def])

In [54]:
print(ast.unparse(ast.fix_missing_locations(main_class_template)))

import numpy as np
import os
from scipy.io import FortranFile

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soiltile = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.veget = np.zeros((self.kj

In [55]:
# Now we modify so that we have attribute depedencies 
new_cls_info, import_nodes1, instance_nodes1 = transformer.create_cls_info(class_def,instance_nodes)

class_module_name = list(new_cls_info.keys())[-1]
class_instance_name = list(new_cls_info[class_module_name].keys())[-1]

In [56]:
new_cls_info[class_module_name]['self'] = new_cls_info[class_module_name].pop(class_instance_name)
print(new_cls_info)

{'Hydrol_vegupd': {'self': {'attributes': {'nsnow': [3, 'int32'], 'nslm': [11, 'int32'], 'nvm': [15, 'int32'], 'nstm': [3, 'int32'], 'kjpindex': [4717, 'int32'], 'ier': [0, 'int32'], 'ic0': [0, 'int32'], 'ic': [0, 'int32'], 'icr': [0.0, 'float64'], 'start_time': [0.0, 'float64'], 'stop_time': [0.0, 'float64'], 'drain_upd': [[{'dim_str': '1', 'dim_end': 'kjpindex'}], 'float64'], 'frac_bare': [[{'dim_str': '1', 'dim_end': 'kjpindex'}, {'dim_str': '1', 'dim_end': 'nvm'}], 'float64'], 'qsintveg': [[{'dim_str': '1', 'dim_end': 'kjpindex'}, {'dim_str': '1', 'dim_end': 'nvm'}], 'float64'], 'runoff_upd': [[{'dim_str': '1', 'dim_end': 'kjpindex'}], 'float64'], 'soiltile': [[{'dim_str': '1', 'dim_end': 'kjpindex'}, {'dim_str': '1', 'dim_end': 'nstm'}], 'float64'], 'veget': [[{'dim_str': '1', 'dim_end': 'kjpindex'}, {'dim_str': '1', 'dim_end': 'nvm'}], 'float64'], 'veget_max': [[{'dim_str': '1', 'dim_end': 'kjpindex'}, {'dim_str': '1', 'dim_end': 'nvm'}], 'float64']}, 'methods': {'__init__': <ast

In [57]:
# Read Template for the vairables within the parent class
read_ast = transformer.prepare_read_code_for_main_template(cls.var_dummy[subroutine_key],assign_nodes)
transformer.correct_function(read_ast,new_cls_info)
identify_replace_all(read_ast.body,new_cls_info)
# print(ast.dump(read_ast,indent=4))
print(ast.unparse(ast.fix_missing_locations(read_ast)))

def read_dummy(self):
    print(f'--- inside the read dummy routine for hydrol_vegupd ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_vegupd/dummy.bin'
    ffile = FortranFile(path, 'r')
    for var in [self.frac_bare, self.qsintveg, self.soiltile, self.veget, self.veget_max]:
        if isinstance(var, np.ndarray):
            arr_shape = var.shape
            if var.dtype == np.float64:
                data = ffile.read_reals(np.float64)
            elif var.dtype == np.int32 or var.dtype == np.bool:
                data = ffile.read_ints(np.int32)
            if data.size != np.prod(arr_shape):
                continue
            var[:] = data.reshape(arr_shape, order='F')


In [58]:
def update_dict(primary_dict, secondary_dict):
    """
    Automatically update the `instances` section of the primary_dict by scanning
    for composed classes (composition) and fetching their attributes and methods
    from the secondary_dict.
    """

    for primary_module, classes in primary_dict.items():
        for primary_class, class_content in classes.items():
            primary_attrs = class_content.get("attributes", {})
            instances = class_content.get("instances", {})

            # For each instance in the primary class, check if it's a composed class
            for instance_name, instance_data in instances.items():
                # Now we must find the class definition of this instance in the secondary_dict
                for secondary_module, secondary_classes in secondary_dict.items():
                    if instance_name in secondary_classes:
                        composed_class = secondary_classes[instance_name]
                        secondary_attrs = composed_class.get("attributes", {})
                        secondary_methods = composed_class.get("methods", {})

                        # Prepare instance sub-structure
                        instance_attrs = instance_data.setdefault("attributes", {})
                        instance_methods = instance_data.setdefault("methods", {})

                        # Add attributes only if not already present in class-level attributes
                        for attr_name, attr_val in secondary_attrs.items():
                            if attr_name not in primary_attrs:
                                instance_attrs.setdefault(attr_name, attr_val)

                        # Add methods (only if not already present in instance-level)
                        for method_name, method_val in secondary_methods.items():
                            instance_methods.setdefault(method_name, method_val)

    return primary_dict


In [59]:
# This was 
new_cls_info = update_dict(
    primary_dict=new_cls_info,
    secondary_dict=cls_info,
)

In [60]:
new_cls_info[class_module_name]['self']['instances']

{'gmhv': {'class_name': <ast.Attribute at 0x7fb4d3727fd0>,
  'attributes': {'numout': [6, 'int32'],
   'min_sechiba': [1e-08, 'float64'],
   'printlev': [2, 'int32'],
   'trois': [3.0, 'float64'],
   'huit': [8.0, 'float64'],
   'zero': [0.0, 'float64'],
   'un': [1.0, 'float64'],
   'ok_bare_soil_new': [False, 'bool'],
   'dz': [[{'dim_str': '1', 'dim_end': 'nslm'}], 'float64'],
   'frac_bare_ns': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
     {'dim_str': '1', 'dim_end': 'nstm'}],
    'float64'],
   'humtot': [[{'dim_str': '1', 'dim_end': 'kjpindex'}], 'float64'],
   'mask_soiltile': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
     {'dim_str': '1', 'dim_end': 'nstm'}],
    'int32'],
   'mask_veget': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
     {'dim_str': '1', 'dim_end': 'nvm'}],
    'int32'],
   'mc': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
     {'dim_str': '1', 'dim_end': 'nslm'},
     {'dim_str': '1', 'dim_end': 'nstm'}],
    'float64'],
   'pref_soil_veg': [[{'dim_str': '1',

In [61]:
# Test function for the parent class 
test_function = transformer.create_test_function(new_cls_info)
transformer.correct_function(test_function,new_cls_info)
print(ast.unparse(ast.fix_missing_locations(test_function)))

def test_hydrol_vegupd(self):
    print('--- inside the test function for hydrol_vegupd ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_vegupd/output.bin'
    ffile = FortranFile(path, 'r')
    modif_var = ['frac_bare', 'frac_bare_ns', 'mask_soiltile', 'mask_veget', 'vegetmax_soil']
    for (variable, value) in zip(modif_var, [self.frac_bare, self.gmhv.frac_bare_ns, self.gmhv.mask_soiltile, self.gmhv.mask_veget, self.gmhv.vegetmax_soil]):
        fortran_value = None
        if isinstance(value, np.ndarray):
            arr_shape = value.shape
            if value.dtype == np.float64:
                fortran_value = ffile.read_reals(np.float64).reshape(arr_shape, order='F')
            elif value.dtype == np.int32 or value.dtype == np.bool:
                fortran_value = ffile.read_ints(np.int32).reshape(arr_shape, order='F')
            if np.allclose(value, fortran_value):
                print(f'Test passed: Variable:{variable} is equal to FORTRAN variable: {variable}')
   

In [62]:
class_def.body.extend([read_ast,test_function]) # Now we add the read dummy for the parent attributes and test method for the parent subroutine

In [63]:
print(cls.actual_arg_spec_list[subroutine_key])
print(cls.dummy_arg_list[subroutine_key])

[['kjpindex', 'veget', 'veget_max', 'soiltile', 'qsintveg', 'frac_bare', 'drain_upd', 'runoff_upd']]
['kjpindex', 'veget', 'veget_max', 'soiltile', 'qsintveg', 'frac_bare', 'drain_upd', 'runoff_upd']


In [64]:
print(cls.actual_arg_spec_list["hydrol_soil_coef"])
print(cls.dummy_arg_list["hydrol_soil_coef"])

[['mcr', 'mcs', 'kjpindex', 'jst', 'njsc'], ['mcr', 'mcs', 'kjpindex', 'jst', 'njsc'], ['mcr', 'mcs', 'kjpindex', 'jst', 'njsc']]
['mcr', 'mcs', 'kjpindex', 'ins', 'njsc']


In [65]:
cls.call_subroutines[child_subroutine_key]

[Call_Stmt(Name('hydrol_tmc_update'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('veget_max'), Name('soiltile'), Name('qsintveg'), Name('drain_upd'), Name('runoff_upd'))))]

In [66]:
cls.all_array_info['hydrol_soil_infilt']

defaultdict(list, {})

In [69]:
new_cls_info[class_module_name]['self']

{'attributes': {'nsnow': [3, 'int32'],
  'nslm': [11, 'int32'],
  'nvm': [15, 'int32'],
  'nstm': [3, 'int32'],
  'kjpindex': [4717, 'int32'],
  'ier': [0, 'int32'],
  'ic0': [0, 'int32'],
  'ic': [0, 'int32'],
  'icr': [0.0, 'float64'],
  'start_time': [0.0, 'float64'],
  'stop_time': [0.0, 'float64'],
  'drain_upd': [[{'dim_str': '1', 'dim_end': 'kjpindex'}], 'float64'],
  'frac_bare': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
    {'dim_str': '1', 'dim_end': 'nvm'}],
   'float64'],
  'qsintveg': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
    {'dim_str': '1', 'dim_end': 'nvm'}],
   'float64'],
  'runoff_upd': [[{'dim_str': '1', 'dim_end': 'kjpindex'}], 'float64'],
  'soiltile': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
    {'dim_str': '1', 'dim_end': 'nstm'}],
   'float64'],
  'veget': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
    {'dim_str': '1', 'dim_end': 'nvm'}],
   'float64'],
  'veget_max': [[{'dim_str': '1', 'dim_end': 'kjpindex'},
    {'dim_str': '1', 'dim_end': 'nvm'}],
  

In [70]:
# print(ast.unparse(ast.fix_missing_locations(new_cls_info[class_module_name]['self']["methods"])))

In [71]:
cls.call_within_sub[subroutine_key]

{'hydrol_tmc_update'}

In [72]:
# Now we can simply call the correct function to ensure that the return stmt, indices and other eleemnts are corrected for both child and 
# parent functions
transformer.cls_mode = True
for child_idx, child_subroutine_key in enumerate(cls.call_within_sub[subroutine_key]):
     transformer.correct_function(module_stacks[child_idx],new_cls_info,None,child_subroutine_key,parent_mode=True)

In [73]:
transformer.correct_function(parent_function_def,new_cls_info,None,subroutine_key)

In [74]:
print(ast.unparse(ast.fix_missing_locations(parent_function_def)))

def hydrol_vegupd(self):
    hydrol_tmc_update()
    mask_veget[:, :] = 0
    mask_soiltile[:, :] = 0
    for jst in range(0, nstm, 1):
        for ji in range(0, kjpindex, 1):
            if soiltile[ji, jst] > min_sechiba:
                mask_soiltile[ji, jst] = 1
    for jv in range(0, nvm, 1):
        for ji in range(0, kjpindex, 1):
            if veget_max[ji, jv] > min_sechiba:
                mask_veget[ji, jv] = 1
    vegetmax_soil[:, :, :] = zero
    for jv in range(0, nvm, 1):
        jst = pref_soil_veg[jv] - 1
        for ji in range(0, kjpindex, 1):
            if mask_soiltile[ji, jst] > 0 and vegtot[ji] > min_sechiba:
                vegetmax_soil[ji, jv, jst] = veget_max[ji, jv] / soiltile[ji, jst]
    for ji in range(0, kjpindex, 1):
        if veget_max[ji, 0] > min_sechiba:
            frac_bare[ji, 0] = un
        else:
            frac_bare[ji, 0] = zero
    if ok_bare_soil_new:
        frac_bare[:, 1:nvm] = zero
    else:
        for jv in range(1, nvm, 1):
    

In [75]:
print(ast.unparse(ast.fix_missing_locations(module_stacks[0])))

def hydrol_tmc_update(self):
    vmr = np.zeros((kjpindex, nstm), dtype=np.float64)
    vmr_sum = np.zeros((kjpindex,), dtype=np.float64)
    delvegtot = np.zeros((kjpindex,), dtype=np.float64)
    mc_dilu = np.zeros((kjpindex, nslm), dtype=np.float64)
    infil_dilu = np.zeros((kjpindex,), dtype=np.float64)
    tmc_old = np.zeros((kjpindex, nstm), dtype=np.float64)
    water2infilt_old = np.zeros((kjpindex, nstm), dtype=np.float64)
    qsintveg_old = np.zeros((kjpindex, nvm), dtype=np.float64)
    test = np.zeros((kjpindex,), dtype=np.float64)
    mcaux = np.zeros((kjpindex, nslm, nstm), dtype=np.float64)
    for ji in range(0, kjpindex, 1):
        if vegtot_old[ji] > min_sechiba:
            for jv in range(0, nvm, 1):
                if veget_max[ji, jv] < min_sechiba and qsintveg[ji, jv] > 0.0:
                    jst = pref_soil_veg[jv] - 1
                    if resdist[ji, jst] > zero:
                        index = jst
                    else:
                        index =

In [76]:
for child_idx in range(len(module_stacks)):
    identify_replace_all(module_stacks[child_idx].body,new_cls_info)
# identify_replace_all(child_function_def.body,new_cls_info)

In [77]:
print(ast.unparse(ast.fix_missing_locations(child_function_def)))

def hydrol_tmc_update(self):
    vmr = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
    vmr_sum = np.zeros((self.kjpindex,), dtype=np.float64)
    delvegtot = np.zeros((self.kjpindex,), dtype=np.float64)
    mc_dilu = np.zeros((self.kjpindex, self.nslm), dtype=np.float64)
    infil_dilu = np.zeros((self.kjpindex,), dtype=np.float64)
    tmc_old = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
    water2infilt_old = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
    qsintveg_old = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
    test = np.zeros((self.kjpindex,), dtype=np.float64)
    mcaux = np.zeros((self.kjpindex, self.nslm, self.nstm), dtype=np.float64)
    for ji in range(0, self.kjpindex, 1):
        if self.gmhv.vegtot_old[ji] > self.gmhv.min_sechiba:
            for jv in range(0, self.nvm, 1):
                if self.veget_max[ji, jv] < self.gmhv.min_sechiba and self.qsintveg[ji, jv] > 0.0:
                    jst = self.gmhv.pref_soil_veg[jv] 

In [78]:
identify_replace_all(parent_function_def.body,new_cls_info)

In [79]:
print(ast.unparse(ast.fix_missing_locations(parent_function_def)))

def hydrol_vegupd(self):
    self.hydrol_tmc_update()
    self.gmhv.mask_veget[:, :] = 0
    self.gmhv.mask_soiltile[:, :] = 0
    for jst in range(0, self.nstm, 1):
        for ji in range(0, self.kjpindex, 1):
            if self.soiltile[ji, jst] > self.gmhv.min_sechiba:
                self.gmhv.mask_soiltile[ji, jst] = 1
    for jv in range(0, self.nvm, 1):
        for ji in range(0, self.kjpindex, 1):
            if self.veget_max[ji, jv] > self.gmhv.min_sechiba:
                self.gmhv.mask_veget[ji, jv] = 1
    self.gmhv.vegetmax_soil[:, :, :] = self.gmhv.zero
    for jv in range(0, self.nvm, 1):
        jst = self.gmhv.pref_soil_veg[jv] - 1
        for ji in range(0, self.kjpindex, 1):
            if self.gmhv.mask_soiltile[ji, jst] > 0 and self.gmhv.vegtot[ji] > self.gmhv.min_sechiba:
                self.gmhv.vegetmax_soil[ji, jv, jst] = self.veget_max[ji, jv] / self.soiltile[ji, jst]
    for ji in range(0, self.kjpindex, 1):
        if self.veget_max[ji, 0] > self.gmhv.mi

In [80]:
init_func = None
for func in ast_walk(class_def, ast.FunctionDef):
    if func.name == '__init__':
        init_func = func

identify_replace_all(init_func.body,new_cls_info)


In [81]:
print(ast.dump(init_func,indent=4))

FunctionDef(
    name='__init__',
    args=arguments(
        posonlyargs=[],
        args=[
            arg(arg='self')],
        kwonlyargs=[],
        kw_defaults=[],
        defaults=[]),
    body=[
        Assign(
            targets=[
                Attribute(
                    value=Name(id='self', ctx=Load()),
                    attr='nsnow',
                    ctx=Store())],
            value=Call(
                func=Attribute(
                    value=Name(id='np', ctx=Load()),
                    attr='int32',
                    ctx=Load()),
                args=[
                    Constant(value=3)],
                keywords=[])),
        Assign(
            targets=[
                Attribute(
                    value=Name(id='self', ctx=Load()),
                    attr='nslm',
                    ctx=Store())],
            value=Call(
                func=Attribute(
                    value=Name(id='np', ctx=Load()),
                    attr='int32',
       

In [82]:
print(ast.unparse(ast.fix_missing_locations(main_class_template)))

import numpy as np
import os
from scipy.io import FortranFile

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soiltile = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.veget = np.zeros((self.kj

In [83]:
### NOW WE use the main template where we will place this inside it 
final_cls_info, _ , _ = transformer.create_cls_info(class_def)
main_template = transformer.out_main_python()
# FIrst we add the global parent module inside the main template 
transformer.insert_at(None, import_nodes[0],main_template)
transformer.insert_at(idx = None,ast_node = class_def,python_template = main_template) 
print(ast.unparse(main_template))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_vegupd

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soilti

In [84]:
main_function_def = [function for function in ast_walk(main_template,ast.FunctionDef) if function.name == "main"]

In [85]:
transformer.add_instance(len(main_function_def[-1].body), instance_nodes1[0], final_cls_info, main_function_def[0], ["read_dummy", subroutine_key, f"test_{subroutine_key}"])

In [86]:
print(ast.unparse(ast.fix_missing_locations(main_template)))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_vegupd

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soilti

In [87]:
# isntance_nodes = transformer.create_instances([class_def],True)
# print(isntance_nodes)

In [88]:
# out_main = transformer.update_main_python(tree,parent_mode=True)

In [89]:
# print(ast.unparse(out_main))

In [90]:
# Where statements 
subroutine_code = """
subroutine compute_c(a, b, c)
  ! Main WHERE block
  WHERE (a > 0)               
    c = b * 2.0   

    WHERE (b < 3.0)      
      c = b + 10.0           
    ELSEWHERE (b >= 3.0)      
      c = b - 1.0            
    END WHERE                 

  ELSEWHERE (a < 0)         
    c = -b                

  ELSEWHERE              
    c = 0.0           

  END WHERE        

end subroutine compute_c
"""

subroutine_code = """
subroutine compute_c(a, b, c, n)
  implicit none
  integer, intent(in) :: n
  real, intent(in) :: a(n), b(n)
  real, intent(out) :: c(n)
  integer :: i

  ! Example loop to process in chunks or apply conditionally
  do i = 1, n
    if (mod(i, 2) == 0) then
      ! Main WHERE block for even indices
      WHERE (a > 0.0)
        c = b * 2.0

        WHERE (b < 3.0)
          c = b + 10.0
        ELSEWHERE (b >= 3.0)
          c = b - 1.0
        END WHERE

      ELSEWHERE (a < 0.0)
        c = -b

      ELSEWHERE
        c = 0.0
      END WHERE

    else
      ! For odd indices, maybe apply a different logic
      WHERE (a < 0.0)
        c = -b * 2.0
      ELSEWHERE
        c = b
      END WHERE
    end if
  end do

end subroutine compute_c
"""
sub_parser = processor.parse_fortran_string(subroutine_code)


INFO     Successfully parsed string!

In [91]:
from f2np import F2NP
f2np = F2NP(cls)

In [92]:
_,_,subroutine_code_ast = f2np.recursive_ast(sub_parser)

In [93]:
print(ast.unparse(ast.fix_missing_locations(subroutine_code_ast[0])))

def compute_c(a, b, c, n):
    for i in range(0, n, 1):
        if np.mod(i, 2) == 0:
            if (a > 0.0).any():
                mask = (a > 0.0)
                c[mask] = b * 2.0
                if (b < 3.0).any():
                    mask = (b < 3.0)
                    c[mask] = b + 10.0
                elif (b >= 3.0).any():
                    mask = (b >= 3.0)
                    c[mask] = b - 1.0
            elif (a < 0.0).any():
                mask = (a < 0.0)
                c[mask] = -b
            else:
                c[mask] = 0.0
        elif (a < 0.0).any():
            mask = (a < 0.0)
            c[mask] = -(b * 2.0)
        else:
            c[mask] = b


In [94]:
main_tree = transformer.update_main_python(out_module=tree,parent_mode=True)

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

...

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Processor initialized.

INFO     Successfully removed INTENT and SAVE attributes from statements

In [95]:
print(ast.unparse(main_tree))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_vegupd

class Hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.gmhv = Global_module_hydrol_vegupd()
        self.gmhv.declaration_initialization()
        self.drain_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.frac_bare = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.qsintveg = np.zeros((self.kjpindex, self.nvm), dtype=np.float64)
        self.runoff_upd = np.zeros((self.kjpindex,), dtype=np.float64)
        self.soilti

In [96]:
transformer.transfer_to_pyfile(main_tree,python_file_type="main")

In [97]:
transformer.transfer_to_pyfile(tree)

In [98]:
print(ast.unparse(ast.fix_missing_locations(tree)))

import numpy as np
import os
from scipy.io import FortranFile

class Global_module_hydrol_vegupd:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.numout = np.int32(6)
        self.min_sechiba = np.float64(1e-08)
        self.printlev = np.int32(2)
        self.trois = np.float64(3.0)
        self.huit = np.float64(8.0)
        self.zero = np.float64(0.0)
        self.un = np.float64(1.0)
        self.ok_bare_soil_new = np.bool(False)
        self.dz = np.zeros((self.nslm,), dtype=np.float64)
        self.frac_bare_ns = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.humtot = np.zeros((self.kjpindex,), dt